## Imports

In [ ]:
import os
import struct
import pandas as pd
from pathlib import Path

## Config

In [ ]:
YUZU_MAIN_SAVE_DIR = Path(os.path.expandvars(r"%APPDATA%\yuzu\nand\user\save"))
RYUJINX_MAIN_SAVE_DIR = Path(os.path.expandvars(r"%APPDATA%\Ryujinx\bis\user\save"))

## Functions

In [45]:
def scan_yuzu_saves(base_path: Path = YUZU_MAIN_SAVE_DIR):
    """
    Scan Yuzu saves by traversing two nested levels (UID → TitleID).
    Returns a DataFrame with TitleID and LastUpdate.
    """
    # First level: UID folder
    uid_folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
    if not uid_folders:
        return pd.DataFrame(columns=["TitleID", "LastUpdate"])
    uid_path = os.path.join(base_path, uid_folders[0])

    # Second level: TitleID folder
    title_folders = [f for f in os.listdir(uid_path) if os.path.isdir(os.path.join(uid_path, f))]
    if not title_folders:
        return pd.DataFrame(columns=["TitleID", "LastUpdate"])
    title_root = os.path.join(uid_path, title_folders[0])

    # Now title_root points to the TitleID folder
    records = []
    for title_folder in os.listdir(title_root):
        full_path = os.path.join(title_root, title_folder)
        if not os.path.isdir(full_path):
            continue
        title_id = os.path.basename(full_path)
        last_update = max(
            (os.path.getmtime(root) for root, _, _ in os.walk(full_path)),
            default=0
        )
        records.append({
            "TitleID": title_id,
            "LastUpdate": pd.to_datetime(last_update, unit="s")
        })

    return pd.DataFrame(records)

In [43]:
def get_ryujinx_title_id(extra_data_path: Path) -> str | None:
    """
    Extract the Title ID from a Ryujinx ExtraData file.
    Reads the first 8 bytes and interprets them as a little-endian unsigned 64-bit integer.
    Returns the Title ID as a 16-character hex string.
    """
    if not extra_data_path.exists():
        return None
    try:
        with open(extra_data_path, "rb") as f:
            data = f.read(8)
            title_id_int = struct.unpack("<Q", data)[0]
            return f"0{title_id_int:015X}"
    except Exception:
        return None

def scan_ryujinx_saves(base_path: Path = RYUJINX_MAIN_SAVE_DIR):
    """
    Scan Ryujinx saves by iterating through each SaveID folder.
    Extract TitleID from ExtraData0 and record last modified time.
    Returns a DataFrame with SaveID, TitleID, and LastUpdate.
    """
    records = []
    for save_folder in base_path.iterdir():
        if not save_folder.is_dir():
            continue

        extra_data_file = save_folder / "ExtraData0"
        title_id = get_ryujinx_title_id(extra_data_file)
        if not title_id:
            continue

        last_update = max(
            (os.path.getmtime(root) for root, _, _ in os.walk(save_folder)),
            default=0
        )
        records.append({
            "SaveID": save_folder.name,
            "TitleID": title_id,
            "LastUpdate": pd.to_datetime(last_update, unit="s")
        })

    return pd.DataFrame(records)

In [44]:
yuzu_df = scan_yuzu_saves()
ryujinx_df = scan_ryujinx_saves()

,SaveID,TitleID,LastUpdate
0,0000000000000001,010069C01AB82000,2025-09-21 16:31:08.306220770
1,0000000000000002,01005CA01580E000,2025-09-21 18:09:26.461978436
2,0000000000000003,010069401ADB8000,2025-07-06 18:57:29.733288050
3,0000000000000004,01003AF0200B0000,2025-06-07 11:54:39.153643608
4,0000000000000005,010000000000100D,2025-06-07 11:39:12.789410353
5,0000000000000006,0100AC20128AC000,2025-09-15 00:10:31.531393290
6,0000000000000007,0100633007D48000,2025-09-22 20:37:54.531503677
7,0000000000000008,0100BEB015604000,2025-11-01 22:10:54.929330587
8,0000000000000009,010025C00D410000,2025-11-01 22:28:02.449795246
9,000000000000000a,0100BF600BF26000,2025-11-22 13:46:46.495593786


In [46]:
yuzu_df

,TitleID,LastUpdate
0,0100000000010000,2023-09-02 17:24:05.439225912
1,010011000EA7A000,2023-09-02 17:24:05.303363085
2,010012F00B6F2000,2023-09-02 17:24:05.219178677
3,010014E00DB56000,2023-09-02 17:24:05.225183725
4,0100192003FA4000,2023-09-02 17:24:05.397726536
5,01001B300B9BE000,2023-09-02 17:24:04.659759998
6,010025C00D410000,2023-09-02 17:24:05.232196569
7,010025F0126FE000,2023-09-02 17:24:05.262226105
8,0100261015BE2000,2024-07-24 17:14:09.453362465
9,01002E7016C46000,2024-03-03 16:38:17.776248455


In [39]:
def compare_saves(ryujinx_df: pd.DataFrame, yuzu_df: pd.DataFrame):
    """
    Merge Ryujinx and Yuzu DataFrames on TitleID.
    Add SyncDirection, CopyFrom, and CopyTo columns to indicate sync actions.
    Uses outer merge to show saves that exist only in one emulator.
    """
    comparison_df = pd.merge(
        ryujinx_df,
        yuzu_df,
        on="TitleID",
        how="outer",
        suffixes=("_Ryujinx", "_Yuzu")
    )

    def determine_direction(row):
        ryujinx_time = row["LastUpdate_Ryujinx"]
        yuzu_time = row["LastUpdate_Yuzu"]

        if pd.isna(ryujinx_time) and not pd.isna(yuzu_time):
            return "Yuzu only"
        elif pd.isna(yuzu_time) and not pd.isna(ryujinx_time):
            return "Ryujinx only"
        elif ryujinx_time > yuzu_time:
            return "Ryujinx → Yuzu"
        elif yuzu_time > ryujinx_time:
            return "Yuzu → Ryujinx"
        else:
            return "Up-to-date"

    def determine_copy(row):
        direction = row["SyncDirection"]
        if direction == "Ryujinx → Yuzu":
            return ("Ryujinx", "Yuzu")
        elif direction == "Yuzu → Ryujinx":
            return ("Yuzu", "Ryujinx")
        elif direction == "Ryujinx only":
            return ("Ryujinx", "Yuzu")
        elif direction == "Yuzu only":
            return ("Yuzu", "Ryujinx")
        else:
            return (None, None)

    comparison_df["SyncDirection"] = comparison_df.apply(determine_direction, axis=1)
    comparison_df[["CopyFrom", "CopyTo"]] = comparison_df.apply(
        lambda row: pd.Series(determine_copy(row)), axis=1
    )

    return comparison_df

# Example usage
comparison_df = compare_saves(ryujinx_df, yuzu_df)

In [40]:
comparison_df

,SaveID,TitleID,LastUpdate_Ryujinx,LastUpdate_Yuzu,SyncDirection,CopyFrom,CopyTo
0,0000000000000005,010000000000100D,2025-06-07 11:39:12.789410353,NaT,Ryujinx only,Ryujinx,Yuzu
1,NaN,0100000000010000,NaT,2023-09-02 17:24:05.439225912,Yuzu only,Yuzu,Ryujinx
2,NaN,010011000EA7A000,NaT,2023-09-02 17:24:05.303363085,Yuzu only,Yuzu,Ryujinx
3,NaN,010012F00B6F2000,NaT,2023-09-02 17:24:05.219178677,Yuzu only,Yuzu,Ryujinx
4,NaN,010014E00DB56000,NaT,2023-09-02 17:24:05.225183725,Yuzu only,Yuzu,Ryujinx
5,NaN,0100192003FA4000,NaT,2023-09-02 17:24:05.397726536,Yuzu only,Yuzu,Ryujinx
6,NaN,01001B300B9BE000,NaT,2023-09-02 17:24:04.659759998,Yuzu only,Yuzu,Ryujinx
7,0000000000000009,010025C00D410000,2025-11-01 22:28:02.449795246,2023-09-02 17:24:05.232196569,Ryujinx → Yuzu,Ryujinx,Yuzu
8,NaN,010025F0126FE000,NaT,2023-09-02 17:24:05.262226105,Yuzu only,Yuzu,Ryujinx
9,NaN,0100261015BE2000,NaT,2024-07-24 17:14:09.453362465,Yuzu only,Yuzu,Ryujinx
